#### Requirement
 1. Load data from flight-time.json into a table
 2. Table structure is given below
```
     FL_DATE DATE, 
     OP_CARRIER STRING, 
     OP_CARRIER_FL_NUM STRING, 
     ORIGIN STRING, 
     ORIGIN_CITY_NAME STRING, 
     DEST STRING, 
     DEST_CITY_NAME STRING, 
     CRS_DEP_TIME LONG, 
     DEP_TIME LONG, 
     WHEELS_ON INT, 
     TAXI_IN INT, 
     CRS_ARR_TIME LONG, 
     ARR_TIME LONG, 
     CANCELLED INT, 
     DISTANCE INT
```
  

####1. Read data from the flight-time.json file

In [0]:
flight_time_raw_df = (
    spark.read.format("json")
    .option("mode","FAILFAST")
    .option("dateFormat","M/d/yyyy")
    .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json')
)

####2. Investigate the dataframe data and schema for problems
In the above cell it is clearly visible the order of attributes doesnt match with the order of table. Attributes are showcased in alphabetical order, also the FL_DATE is shown as string which we expect it to be as Date type in our Target table. For resolving such issues will have to define a schema which our JSON connector can follow

In [0]:
flight_time_raw_df.limit(3).display()

####3. Define Dataframe schema before reading it

In [0]:
from pyspark.sql.types import DateType, StringType, StructField, StructType, IntegerType, LongType

flight_schema = StructType([
    StructField("FL_DATE", DateType()),
    StructField("OP_CARRIER", StringType()),
    StructField("OP_CARRIER_FL_NUM", StringType()),
    StructField("ORIGIN", StringType()),
    StructField("ORIGIN_CITY_NAME", StringType()),
    StructField("DEST", StringType()),
    StructField("DEST_CITY_NAME", StringType()),
    StructField("CRS_DEP_TIME", LongType()),
    StructField("DEP_TIME", LongType()),
    StructField("WHEELS_ON", IntegerType()),
    StructField("TAXI_IN", IntegerType()),
    StructField("CRS_ARR_TIME", LongType()),
    StructField("ARR_TIME", LongType()),
    StructField("CANCELLED", IntegerType()),
    StructField("DISTANCE", IntegerType())
])

####4. Read datafile with schema-on-read

In [0]:
flight_time_raw_with_schema_df = (
    spark.read.format("json")
    .option("mode","FAILFAST")
    .option("dateFormat","M/d/yyyy")
    .schema(flight_schema)
    .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/flight-time.json')
)

####5. Save the Dataframe to the table flight_time_raw

In [0]:
flight_time_raw_with_schema_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time_raw")

####Requirement
1. Read raw data from flight_time_raw table
 2. Apply transformations to time values as hour to minute interval

     1. CRS_DEP_TIME
     2. DEP_TIME
     3. WHEELS_ON
     4. CRS_ARR_TIME
     5. ARR_TIME
 3. Apply transformation to TAXI_IN to make it a minute interval

####1. Read data to create a dataframe

In [0]:
flight_time_raw_df = (
    spark.read.table('dev.spark_db.flight_time_raw')
)

####2. Develop logic to transform CRS_DEP_TIME to an interval

In [0]:
from pyspark.sql.functions import expr
#withColumns allow us to create new columns or modify existing columns
step1_df = (flight_time_raw_df.withColumns(
    {
        "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'),2)"),  
        #lpad used for padding, 4 is the length of the string, '0' is the character to be padded
        "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'),2)") 
        #CRS_DEP_TIME_MM column is created but if it would have been CRS_DEP_TIME it would have been overwritten
    }
)
)

step2_df = (step1_df.withColumns(
    {
        "CRS_DEP_TIME_NEW": expr("cast(concat(CRS_DEP_TIME_HH,':',CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)") #INTERVAL HOUR TO MINUTE is a spark datatype. Cast is used to convert the string to a spark datatype
    }
))

####3. Develop a reusable function

In [0]:
def get_interval(hhmm):
    from pyspark.sql.functions import expr

    return expr(f"""
                cast(concat(left(lpad({hhmm}, 4, '0'),2),':',
                right(lpad({hhmm}, 4, '0'),2))
                AS INTERVAL HOUR TO MINUTE)
                """)

####4. Apply function to dataframe

In [0]:
result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME" : get_interval("CRS_DEP_TIME"),
        "DEP_TIME" : get_interval("DEP_TIME"),
        "WHEELS_ON" : get_interval("WHEELS_ON"),
        "CRS_ARR_TIME" : get_interval("CRS_ARR_TIME"),
        "ARR_TIME" : get_interval("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

####5. Save results to the table 

In [0]:
result_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time")

### Exploratory Data Analysis and type correction

####1. Read data from the sales_sample.csv file and analyse to identify problems

1.1 Define schema

In [0]:
file_schema = """
id int,
name string,
dop string,
phone long,
amount string,
discount string
"""

1.2 Read data

In [0]:
sales_raw_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(file_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
)
sales_raw_df.display()


1.3 Describe the data

In [0]:
sales_raw_df.describe().display()

1.4 List down the problems you want to fix
 1. Convert id from integer to string and rename it as transaction_id.
 2. Rename the name column to customer_name.
 3. Convert the dop to date format and rename the column to date_of_purchase.
 4. Rename the phone column to customer_phone
 5. Convert the amount to a long value and filter out nulls and outlier values. 
 6. Rename the column to purchase_amount
 7. Convert discount to double, converting nil and null values to zero. rename the column to applied_discount

####2. Prepare and clean the Dataframe using appropriate transformations

2.1 Transform

In [0]:
sales_df = sales_raw_df.selectExpr(
    "cast(id as string) as transaction_id",
    "name as customer_name",
    "nvl(try_cast(dop as date), to_date(dop,'dd-MM-yy')) as date_of_purchase ",
    "cast(phone as string) as customer_phone",
    "cast(amount as long) as purchase_amount ",
    "nvl(try_cast(discount as double),0) as applied_discount "
).filter("purchase_amount is not null and purchase_amount < 200000")

sales_df.display()

2.2 Verify statistics

In [0]:
sales_df.describe().display()